# Speaker Diarization with pyannote and OpenVINO

Speaker diarization answers the question *"who spoke when?"* by partitioning a recording into speech segments and assigning each segment to a speaker, without knowing the speakers in advance. It is a common building block for meeting transcription, call-center analytics, and preparing training data for speech models.

This notebook uses the open-source [`pyannote/speaker-diarization-community-1`](https://huggingface.co/pyannote/speaker-diarization-community-1) pipeline and accelerates it with OpenVINO. It is a modern replacement for the speaker-diarization notebook that was deprecated in 2024, based on the newer `community-1` pipeline.

The pipeline has three stages:

- **segmentation** ([`pyannote/segmentation-3.0`](https://huggingface.co/pyannote/segmentation-3.0), a PyanNet model) — detects speech and overlapped speech,
- **speaker embedding** (a WeSpeaker ResNet34 model) — turns speech into speaker vectors,
- **clustering** — groups the vectors into speakers.

The two heavy neural blocks (segmentation and the embedding ResNet) are converted to OpenVINO IR and run on the OpenVINO device you select (CPU or GPU), while pyannote keeps orchestrating windowing and clustering in Python.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Hugging Face access](#Hugging-Face-access)
- [Load the diarization pipeline](#Load-the-diarization-pipeline)
- [Download a sample audio file](#Download-a-sample-audio-file)
- [Run diarization with PyTorch](#Run-diarization-with-PyTorch)
- [Convert the pipeline to OpenVINO](#Convert-the-pipeline-to-OpenVINO)
    - [Export the neural blocks to OpenVINO IR](#Export-the-neural-blocks-to-OpenVINO-IR)
    - [Select inference device](#Select-inference-device)
    - [Run diarization with OpenVINO](#Run-diarization-with-OpenVINO)
- [Optional: run diarization on Intel XPU](#Optional:-run-diarization-on-Intel-XPU)
- [Optional: full VoxConverse DER benchmark](#Optional:-full-VoxConverse-DER-benchmark)
- [Cleanup](#Cleanup)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/pyannote-audio/pyannote-audio.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Install the required packages. `pyannote.audio` provides the diarization pipeline, and OpenVINO accelerates the neural blocks.


In [ ]:
import platform

%pip install -q "openvino>=2025.1.0"
%pip install -q "pyannote.audio>=4.0.0" "onnx" "soundfile" "librosa" "ipywidgets" "torch>=2.4.0" "torchaudio" --extra-index-url https://download.pytorch.org/whl/cpu

import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("pyannote-audio.ipynb")


## Hugging Face access
[back to top ⬆️](#Table-of-contents:)

`pyannote/speaker-diarization-community-1` is a **gated** model. Before running this notebook you need to:

1. Create a *Read* access token at https://huggingface.co/settings/tokens.
2. Accept the user conditions of both models used by the pipeline:
   - https://huggingface.co/pyannote/speaker-diarization-community-1
   - https://huggingface.co/pyannote/segmentation-3.0

Then log in with your token. The token is stored locally by `huggingface_hub` and reused on later runs.


In [ ]:
from huggingface_hub import get_token, notebook_login

if get_token() is None:
    notebook_login()
else:
    print("Already logged in to Hugging Face.")


## Load the diarization pipeline
[back to top ⬆️](#Table-of-contents:)

`Pipeline.from_pretrained` downloads the pipeline definition and its model checkpoints (cached after the first run). pyannote orchestration stays on CPU PyTorch; only the neural blocks are moved to OpenVINO later.


In [ ]:
import torch
from pyannote.audio import Pipeline

PIPELINE_ID = "pyannote/speaker-diarization-community-1"

pipeline = Pipeline.from_pretrained(PIPELINE_ID)
if pipeline is None:
    raise RuntimeError(
        f"Could not load '{PIPELINE_ID}'. Accept the model conditions on Hugging Face "
        "and make sure you are logged in (see the previous cell)."
    )
pipeline.to(torch.device("cpu"))
print("Pipeline loaded.")


## Download a sample audio file
[back to top ⬆️](#Table-of-contents:)

We use the short multi-speaker sample shipped with the `pyannote.audio` tutorials.


In [ ]:
import IPython.display as ipd

from notebook_utils import download_file

audio_file = Path("sample.wav")
if not audio_file.exists():
    download_file(
        "https://github.com/pyannote/pyannote-audio/raw/develop/tutorials/assets/sample.wav",
        filename=audio_file.name,
    )

ipd.Audio(str(audio_file))


## Run diarization with PyTorch
[back to top ⬆️](#Table-of-contents:)

First run the pipeline as-is on PyTorch to get a baseline result. The helper below prints each speaker turn and the detected number of speakers.


In [ ]:
import time


def show_diarization(diarization) -> None:
    """Print each speaker turn and the total number of detected speakers."""
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        print(f"start={turn.start:6.1f}s stop={turn.end:6.1f}s {speaker}")
    speakers = diarization.labels()
    print(f"\n{len(speakers)} speaker(s): {', '.join(speakers)}")


start = time.perf_counter()
torch_output = pipeline(str(audio_file))
print(f"PyTorch diarization took {time.perf_counter() - start:.1f}s\n")

show_diarization(torch_output.speaker_diarization)


## Convert the pipeline to OpenVINO
[back to top ⬆️](#Table-of-contents:)

The pipeline spends most of its time in two neural blocks: the segmentation model (`PyanNet`) and the speaker-embedding ResNet. We convert both to OpenVINO IR and route their `forward` calls through the OpenVINO runtime. The embedding model's filter-bank (fbank) front end uses `torch.vmap` + Kaldi ops that cannot be traced, so it stays in PyTorch while the ResNet part runs on OpenVINO.


### Export the neural blocks to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

Trace each block and save it as OpenVINO IR (`.xml` / `.bin`) into `ov_models/`. IR weights are stored in FP16 by default (`ov.save_model(..., compress_to_fp16=True)`).


In [ ]:
from unittest.mock import MagicMock

import openvino as ov

OV_MODEL_DIR = Path("ov_models")
OV_MODEL_DIR.mkdir(exist_ok=True)
SEGMENTATION_XML = OV_MODEL_DIR / "segmentation.xml"
EMBEDDING_XML = OV_MODEL_DIR / "embedding_resnet.xml"


def prepare_for_trace(model: torch.nn.Module) -> None:
    """Attach a dummy trainer so pyannote Lightning modules can be traced."""
    model.eval()
    for module in model.modules():
        module._trainer = MagicMock()


def export_segmentation(pipeline: Pipeline, xml_path: Path) -> None:
    """Convert the segmentation model to OpenVINO IR with a dynamic batch."""
    inference = pipeline._segmentation
    model = inference.model
    prepare_for_trace(model)
    num_samples = int(inference.duration * model.audio.sample_rate)
    example = torch.randn(1, 1, num_samples)
    with torch.inference_mode():
        traced = torch.jit.trace(model, example, strict=False, check_trace=False)
    ov_model = ov.convert_model(traced, example_input=example, input=[ov.PartialShape([-1, 1, num_samples])])
    ov.save_model(ov_model, xml_path)


def export_embedding_resnet(pipeline: Pipeline, xml_path: Path) -> None:
    """Convert the embedding ResNet to OpenVINO IR (fbank stays in PyTorch)."""
    model = pipeline._embedding.model_
    prepare_for_trace(model)
    with torch.inference_mode():
        fbank = model.compute_fbank(torch.randn(2, 1, 32000))
    num_mels = fbank.shape[2]

    class ResnetWrap(torch.nn.Module):
        def __init__(self, resnet: torch.nn.Module):
            super().__init__()
            self.resnet = resnet

        def forward(self, fbank, weights):
            return self.resnet(fbank, weights=weights)[1]

    wrap = ResnetWrap(model.resnet).eval()
    example = (torch.randn(2, fbank.shape[1], num_mels), torch.ones(2, 50))
    with torch.inference_mode():
        traced = torch.jit.trace(wrap, example, strict=False, check_trace=False)
    ov_model = ov.convert_model(
        traced,
        example_input=example,
        input=[ov.PartialShape([-1, -1, num_mels]), ov.PartialShape([-1, -1])],
    )
    ov.save_model(ov_model, xml_path)


if not (SEGMENTATION_XML.exists() and EMBEDDING_XML.exists()):
    export_segmentation(pipeline, SEGMENTATION_XML)
    export_embedding_resnet(pipeline, EMBEDDING_XML)
print(f"OpenVINO IR is ready in '{OV_MODEL_DIR}'.")


### Select inference device
[back to top ⬆️](#Table-of-contents:)

Select the OpenVINO device for inference from the dropdown. Choose `CPU`, `GPU` (if you have an Intel GPU), or `AUTO`.


In [ ]:
import time

import numpy as np
import openvino as ov

from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])

# Show the available devices and the current selection.
core = ov.Core()
print("Available OpenVINO devices:")
for d in core.available_devices:
    print(f"  {d}: {core.get_property(d, 'FULL_DEVICE_NAME')}")


def resolve_device(dev: str) -> str:
    """Report the physical device the selection maps to.

    AUTO first runs on a CPU stub (reported as "(CPU)") while the target device
    compiles in the background, so we keep inferring until the device settles.
    """
    compiled = core.compile_model(core.read_model(SEGMENTATION_XML), dev)
    req = compiled.create_infer_request()
    inputs = [np.zeros([d.get_length() if d.is_static else 1 for d in inp.get_partial_shape()], np.float32) for inp in compiled.inputs]
    executed = list(compiled.get_property("EXECUTION_DEVICES"))
    deadline = time.time() + 10
    while time.time() < deadline:
        req.infer(inputs)
        executed = list(compiled.get_property("EXECUTION_DEVICES"))
        # Parentheses mark the transient CPU fallback; wait for the real device.
        if not any("(" in e for e in executed):
            break
        time.sleep(0.3)
    return ", ".join(e.strip("()") for e in executed)


print(f"\nSelected device: {device.value} -> running on: {resolve_device(device.value)}")

device


### Run diarization with OpenVINO
[back to top ⬆️](#Table-of-contents:)

The functions below replace each block's `forward` with an OpenVINO-backed one. On GPU, dynamic input shapes are slow, so we reshape the IR to fixed batch "buckets" (padding each call up to the next power of two) — this is much faster on Intel GPUs and avoids over-padding. On CPU we keep dynamic shapes.


In [ ]:
import numpy as np


def bucket_size(batch: int, max_batch: int) -> int:
    """Return the smallest power of two >= batch, capped at max_batch."""
    size = 1
    while size < batch:
        size *= 2
    return min(size, max_batch)


def accelerate_segmentation(pipeline: Pipeline, core: ov.Core, device: str, static: bool) -> None:
    """Route the segmentation model's forward through the exported OpenVINO IR."""
    model = pipeline._segmentation.model

    if not static:
        compiled = core.compile_model(core.read_model(SEGMENTATION_XML), device)
        out_port = compiled.output(0)

        def forward(waveforms, *args, **kwargs):
            data = waveforms.detach().cpu().numpy().astype(np.float32)
            return torch.from_numpy(compiled(data)[out_port])

        model.forward = forward
        return

    max_batch = pipeline._segmentation.batch_size
    buckets: dict = {}

    def forward(waveforms, *args, **kwargs):
        data = waveforms.detach().cpu().numpy().astype(np.float32)
        batch = data.shape[0]
        size = bucket_size(batch, max_batch)
        if size not in buckets:
            ov_model = core.read_model(SEGMENTATION_XML)
            ov_model.reshape([size, data.shape[1], data.shape[2]])
            compiled = core.compile_model(ov_model, device)
            buckets[size] = (compiled, compiled.output(0))
        compiled, out_port = buckets[size]
        if batch < size:
            data = np.concatenate([data, np.zeros((size - batch, *data.shape[1:]), np.float32)], axis=0)
        return torch.from_numpy(compiled(data)[out_port][:batch].copy())

    model.forward = forward


def accelerate_embedding(pipeline: Pipeline, core: ov.Core, device: str, static: bool) -> None:
    """Route the embedding ResNet through OpenVINO IR; keep fbank in PyTorch."""
    model = pipeline._embedding.model_
    resnet = model.resnet  # PyTorch fallback for non-2D-weight calls

    if not static:
        compiled = core.compile_model(core.read_model(EMBEDDING_XML), device)
        out_port = compiled.output(0)

        def forward(waveforms, weights=None, *args, **kwargs):
            fbank = model.compute_fbank(waveforms.detach().cpu())
            if weights is None or weights.ndim != 2:
                return resnet(fbank, weights=weights)[1]
            result = compiled((fbank.numpy().astype(np.float32), weights.detach().cpu().numpy().astype(np.float32)))[out_port]
            return torch.from_numpy(result)

        model.forward = forward
        return

    max_batch = pipeline.embedding_batch_size
    buckets: dict = {}

    def forward(waveforms, weights=None, *args, **kwargs):
        fbank = model.compute_fbank(waveforms.detach().cpu())
        if weights is None or weights.ndim != 2:
            return resnet(fbank, weights=weights)[1]
        fb = fbank.numpy().astype(np.float32)
        wt = weights.detach().cpu().numpy().astype(np.float32)
        batch, frames, mels = fb.shape
        wframes = wt.shape[1]
        size = bucket_size(batch, max_batch)
        key = (size, frames, wframes)
        if key not in buckets:
            ov_model = core.read_model(EMBEDDING_XML)
            ov_model.reshape({0: [size, frames, mels], 1: [size, wframes]})
            compiled = core.compile_model(ov_model, device)
            buckets[key] = (compiled, compiled.output(0))
        compiled, out_port = buckets[key]
        if batch < size:
            fb = np.concatenate([fb, np.zeros((size - batch, frames, mels), np.float32)], axis=0)
            # pad weights with ones (not zeros) so stats-pooling never divides by zero
            wt = np.concatenate([wt, np.ones((size - batch, wframes), np.float32)], axis=0)
        return torch.from_numpy(compiled((fb, wt))[out_port][:batch].copy())

    model.forward = forward


core = ov.Core()
ov_device = device.value
# GPUs are very slow with dynamic shapes (they recompile per shape), so use the
# static bucketed path whenever a GPU will run inference - either an explicit GPU
# selection or AUTO on a machine that has a GPU.
static = "GPU" in ov_device or (ov_device == "AUTO" and any("GPU" in d for d in core.available_devices))

accelerate_segmentation(pipeline, core, ov_device, static)
accelerate_embedding(pipeline, core, ov_device, static)
print(f"Pipeline neural blocks now run on OpenVINO device: {ov_device} (static shapes: {static})")


Now run the accelerated pipeline on the selected OpenVINO device and print the result. Compare the runtime and speaker turns with the PyTorch baseline above.


In [ ]:
start = time.perf_counter()
ov_output = pipeline(str(audio_file))
print(f"OpenVINO ({ov_device}) diarization took {time.perf_counter() - start:.1f}s\n")

show_diarization(ov_output.speaker_diarization)


## Optional: run diarization on Intel XPU
[back to top ⬆️](#Table-of-contents:)

As an alternative to OpenVINO, the pipeline can also run on an Intel GPU through PyTorch's **XPU** backend. This path keeps the model in PyTorch and offloads it to the Intel GPU with `pipeline.to("xpu")`.

This section is **optional** and only runs when an Intel XPU device is detected (it needs the XPU build of PyTorch). On machines without an XPU — including CI — it is skipped automatically.

To enable it on a Linux machine with an Intel GPU, set `INSTALL_XPU = True` in the next cell to install the XPU build of PyTorch, **restart the kernel**, then re-run the notebook.


In [ ]:
# The XPU backend needs the Intel XPU build of PyTorch (Linux + Intel GPU only).
# It replaces the CPU torch/torchaudio in this kernel, so RESTART THE KERNEL after
# installing and then re-run the notebook. Leave this False on CPU-only machines
# and in CI.
INSTALL_XPU = False

import platform
import subprocess
import sys

import torch

if "+xpu" in torch.__version__:
    print(f"XPU build already installed: torch {torch.__version__} - nothing to do.")
elif INSTALL_XPU and platform.system() == "Linux":
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--force-reinstall", "torch", "torchaudio",
         "--index-url", "https://download.pytorch.org/whl/xpu"]
    )
    print("\nXPU PyTorch installed. Please RESTART THE KERNEL, then run the notebook again.")
else:
    print("XPU install skipped. Set INSTALL_XPU = True on a Linux machine with an Intel GPU to install it.")


### Run diarization on the Intel XPU
[back to top ⬆️](#Table-of-contents:)

With the XPU build of PyTorch installed and the kernel restarted, run the pipeline on the Intel GPU. A fresh pipeline is used so the OpenVINO `forward` patches are not reused. The cell auto-skips when no XPU device is present.


In [ ]:
RUN_XPU = True  # only runs when an Intel XPU device is actually present

if RUN_XPU and hasattr(torch, "xpu") and torch.xpu.is_available():
    print(f"Intel XPU detected: {torch.xpu.get_device_name(0)}\n")
    try:
        # Use a fresh pipeline so the OpenVINO forward patches above are not reused.
        xpu_pipeline = Pipeline.from_pretrained(PIPELINE_ID)
        xpu_pipeline.to(torch.device("xpu"))

        start = time.perf_counter()
        xpu_output = xpu_pipeline(str(audio_file))
        print(f"PyTorch XPU diarization took {time.perf_counter() - start:.1f}s\n")
        show_diarization(xpu_output.speaker_diarization)
    except Exception as error:
        print(f"XPU run failed: {error}")
elif RUN_XPU:
    print("No Intel XPU available - skipping. Install the XPU build of PyTorch and run on an Intel GPU to enable it.")
else:
    print("XPU run disabled. Set RUN_XPU = True to try it.")


## Optional: full VoxConverse DER benchmark
[back to top ⬆️](#Table-of-contents:)

The cells below measure Diarization Error Rate (DER) on the [VoxConverse](https://github.com/joonson/voxconverse) test set. This is **disabled by default** because it downloads several GB of audio and takes much longer than the five-minute target for the rest of the notebook.

Set `RUN_FULL_BENCHMARK = True` in the next cell to automatically download the VoxConverse test set (RTTM references from the official GitHub repo and test audio from the Oxford VGG mirror) and run the benchmark. The download is cross-platform (pure Python) and only happens once; existing files are reused.

**Note on scoring:** DER is computed over the **whole recording**, which is the standard VoxConverse convention. Restricting the evaluation map (UEM) to the reference speech timeline (for example `annotation.get_timeline().support()`) would exclude false alarms during silence and report an optimistically low DER.


In [ ]:
RUN_FULL_BENCHMARK = False  # set True to download VoxConverse and run the DER benchmark
VOX_ROOT = Path("voxconverse")  # dataset root populated by the download step below

if RUN_FULL_BENCHMARK:
    import io
    import zipfile

    from notebook_utils import download_file

    VOX_ROOT.mkdir(exist_ok=True)
    rttm_dir = VOX_ROOT / "test"
    wav_dir = VOX_ROOT / "voxconverse_test_wav"

    # 1. RTTM reference annotations (VoxConverse v0.3) from the official GitHub repo.
    if not any(rttm_dir.glob("*.rttm")) if rttm_dir.exists() else True:
        print("Downloading VoxConverse RTTM annotations...")
        rttm_dir.mkdir(parents=True, exist_ok=True)
        repo_zip = requests.get(
            "https://github.com/joonson/voxconverse/archive/refs/heads/master.zip", timeout=60
        ).content
        with zipfile.ZipFile(io.BytesIO(repo_zip)) as zf:
            for name in zf.namelist():
                if "/test/" in name and name.endswith(".rttm"):
                    (rttm_dir / Path(name).name).write_bytes(zf.read(name))
    print(f"RTTM reference files: {len(list(rttm_dir.glob('*.rttm')))}")

    # 2. Test audio (~2 GB) from the Oxford VGG mirror.
    if not (wav_dir.exists() and any(wav_dir.rglob("*.wav"))):
        print("Downloading VoxConverse test audio (~2 GB, this can take a while)...")
        audio_zip = download_file(
            "https://www.robots.ox.ac.uk/~vgg/data/voxconverse/data/voxconverse_test_wav.zip",
            directory=VOX_ROOT,
        )
        with zipfile.ZipFile(audio_zip) as zf:
            zf.extractall(wav_dir)
    print(f"Test audio files: {len(list(wav_dir.rglob('*.wav')))}")
    print("VoxConverse test set ready.")
else:
    print("Full VoxConverse benchmark disabled. Set RUN_FULL_BENCHMARK = True to download and run it.")


### Score DER on the VoxConverse test set
[back to top ⬆️](#Table-of-contents:)

Run the diarization pipeline on every test file and accumulate the Diarization Error Rate. Progress is printed per file as `[i/total] <uri>: DER=...%`, followed by the overall `TOTAL DER`.


In [ ]:
if RUN_FULL_BENCHMARK:
    import soundfile as sf
    from pyannote.core import Segment, Timeline
    from pyannote.database.util import load_rttm
    from pyannote.metrics.diarization import DiarizationErrorRate

    print(f"Scoring with OpenVINO-accelerated pipeline on device: {ov_device} (static shapes: {static})\n")
    
    annotations: dict = {}
    for rttm_path in sorted((VOX_ROOT / "test").glob("*.rttm")):
        annotations.update(load_rttm(rttm_path))

    wav_index = {p.stem: p for p in VOX_ROOT.rglob("*.wav")}
    metric = DiarizationErrorRate(collar=0.0, skip_overlap=False)

    total = len(annotations)
    for i, (uri, reference) in enumerate(sorted(annotations.items()), 1):
        audio_path = wav_index.get(uri)
        if audio_path is None:
            continue
        hypothesis = pipeline(str(audio_path)).speaker_diarization
        # Score over the WHOLE recording (VoxConverse convention): the UEM spans
        # the full audio so false alarms during silence are counted.
        uem = Timeline([Segment(0.0, sf.info(str(audio_path)).duration)])
        der = metric(reference, hypothesis, uem=uem, uri=uri)
        print(f"[{i}/{total}] {uri}: DER={100 * der:.1f}%", flush=True)

    report = metric.report(display=False)
    total_der = report.loc["TOTAL", ("diarization error rate", "%")]
    print(f"\nTOTAL DER = {float(total_der):.1f}%")
else:
    print("Full VoxConverse benchmark skipped. Set RUN_FULL_BENCHMARK = True to run it.")


## Cleanup
[back to top ⬆️](#Table-of-contents:)

Uncomment the lines below to remove the downloaded sample and the exported OpenVINO IR. Set `REMOVE_XPU = True` to also uninstall the Intel XPU build of PyTorch and its runtime packages and restore the CPU build (restart the kernel afterwards).


In [ ]:
import shutil

audio_file.unlink(missing_ok=True)
shutil.rmtree(OV_MODEL_DIR, ignore_errors=True)
Path("notebook_utils.py").unlink(missing_ok=True)

# Optional: undo the Intel XPU install and restore the CPU build of PyTorch.
# Set REMOVE_XPU = True, run this cell, then RESTART THE KERNEL.
REMOVE_XPU = False

if REMOVE_XPU:
    import subprocess
    import sys

    xpu_packages = [
        "torch", "torchaudio", "triton-xpu",
        "intel-sycl-rt", "intel-opencl-rt", "intel-openmp", "intel-pti",
        "intel-cmplr-lib-rt", "intel-cmplr-lib-ur", "intel-cmplr-lic-rt",
        "dpcpp-cpp-rt", "mkl", "oneccl", "oneccl-devel", "onemkl-license",
        "onemkl-sycl-blas", "onemkl-sycl-dft", "onemkl-sycl-lapack",
        "onemkl-sycl-rng", "onemkl-sycl-sparse", "tbb", "tcmlib", "umf",
        "impi-rt", "pyzes",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", *xpu_packages])
    # Restore the CPU build so the rest of the notebook still works.
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "torch", "torchaudio",
         "--index-url", "https://download.pytorch.org/whl/cpu"]
    )
    print("\nXPU packages removed and CPU PyTorch restored. Please RESTART THE KERNEL.")
